In [44]:
import pandas as pd
import numpy as np

In [45]:
df = pd.read_csv("dados/confrontos/historico_confrontos.csv")
display(df.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [46]:
df = df.drop(columns=["tournament","city","country","neutral"])

In [47]:
from datetime import date

def convert_data_string(data_string: str):
    year, month, day = list(map(int, data_string.split("-")))
    return date(year,month,day)

df["date"] = df["date"].apply(convert_data_string)

display(df.head())

,date,home_team,away_team,home_score,away_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0


In [48]:
fim_copa_22 = date(year=2022,month=12, day=18)
inicio_copa26 = date(year=2026,month=6,day=11)
df_ciclo26 = df[(df["date"] > fim_copa_22) & (df["date"] < inicio_copa26)]

display(df_ciclo26)

,date,home_team,away_team,home_score,away_score
45787,2022-12-20,Cambodia,Philippines,3.0,2.0
45788,2022-12-20,Brunei,Thailand,0.0,5.0
45789,2022-12-21,Myanmar,Malaysia,0.0,1.0
45790,2022-12-21,Laos,Vietnam,0.0,6.0
45791,2022-12-23,Oman,Syria,2.0,1.0
...,...,...,...,...,...
49400,2026-06-09,Iraq,Venezuela,0.0,2.0
49401,2026-06-10,Bolivia,Algeria,0.0,4.0
49402,2026-06-10,England,Costa Rica,3.0,0.0
49403,2026-06-10,Portugal,Nigeria,2.0,1.0


In [49]:
qualified_teams = [
    "Canada",
    "United States",
    "Mexico",
    "Curacao",
    "Haiti",
    "Panama",
    "Japan",
    "Iran",
    "Uzbekistan",
    "South Korea",
    "Jordan",
    "Australia",
    "Qatar",
    "Saudi Arabia",
    "New Zealand",
    "Argentina",
    "Brazil",
    "Ecuador",
    "Uruguay",
    "Colombia",
    "Paraguay",
    "Morocco",
    "Tunisia",
    "Egypt",
    "Algeria",
    "Ghana",
    "Cape Verde",
    "South Africa",
    "Ivory Coast",
    "Senegal",
    "England",
    "France",
    "Croatia",
    "Portugal",
    "Norway",
    "Netherlands",
    "Germany",
    "Switzerland",
    "Austria",
    "Belgium",
    "Spain",
    "Scotland",
    "Turkey",
    "Czech Republic",
    "Sweden",
    "Bosnia and Herzegovina",
    "DR Congo",
    "Iraq"
]

df_ciclo26 = df_ciclo26[(df["home_team"].isin(qualified_teams)) | (df["away_score"].isin(qualified_teams))]
df_ciclo26 = df_ciclo26.reset_index()

/tmp/ipykernel_48507/3298589592.py:52: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ciclo26 = df_ciclo26[(df["home_team"].isin(qualified_teams)) | (df["away_score"].isin(qualified_teams))]


In [50]:
print(df_ciclo26.shape)

(1091, 6)


In [51]:
display(df_ciclo26)

,index,date,home_team,away_team,home_score,away_score
0,45802,2022-12-30,Iraq,Kuwait,1.0,0.0
1,45811,2023-01-06,Iraq,Oman,0.0,0.0
2,45818,2023-01-09,Sweden,Finland,2.0,0.0
3,45820,2023-01-09,Iraq,Saudi Arabia,2.0,0.0
4,45823,2023-01-10,Qatar,Bahrain,1.0,2.0
...,...,...,...,...,...,...
1086,49386,2026-06-09,DR Congo,Chile,1.0,2.0
1087,49392,2026-06-09,Saudi Arabia,Senegal,0.0,0.0
1088,49400,2026-06-09,Iraq,Venezuela,0.0,2.0
1089,49402,2026-06-10,England,Costa Rica,3.0,0.0


In [52]:
df_ciclo26.to_csv("dados/confrontos/historico_confrontos_ciclo2026.csv")

anos_copa = [i for i in range(1930,2023,4)]
anos_copa.remove(1946)
anos_copa.remove(1942)

In [53]:
import os

df_todos_jogos = pd.read_csv("dados/confrontos/historico_confrontos.csv")

df_todos_jogos['date'] = pd.to_datetime(df_todos_jogos['date'])

os.makedirs("dados/confrontos", exist_ok=True)

def separar_jogos_por_ciclo(ano):
    """
    Filtra todos os jogos do mundo que ocorreram no intervalo de 4 anos 
    antes da Copa especificada.
    """
    
    if ano == 1930:
        data_inicio = pd.to_datetime("1926-08-01")
        data_fim = pd.to_datetime("1930-06-30")
        
    elif ano == 1950:
        data_inicio = pd.to_datetime("1946-08-01")
        data_fim = pd.to_datetime("1950-05-31")
        
    elif ano == 2022:
        data_inicio = pd.to_datetime("2018-08-01")
        data_fim = pd.to_datetime("2022-10-31")
        
    else:
        data_inicio = pd.to_datetime(f"{ano-4}-08-01")
        data_fim = pd.to_datetime(f"{ano}-05-31")
        
    mask = (df_todos_jogos['date'] >= data_inicio) & (df_todos_jogos['date'] <= data_fim)
    df_ciclo = df_todos_jogos[mask].copy()

    df_ciclo["resultado"] = np.where(
        df_ciclo["home_score"] > df_ciclo["away_score"], 1, 
        np.where(df_ciclo["away_score"] > df_ciclo["home_score"], 2, 0)  
    )
    
    caminho_saida = f"dados/confrontos/historico_confrontos_ciclo{ano}.csv"
    df_ciclo.to_csv(caminho_saida, index=False)
    
    print(f"Ciclo {ano}: {len(df_ciclo)} jogos salvos (De {data_inicio.date()} até {data_fim.date()})")

# Executar a função para todos os anos
print("Iniciando a separação dos ciclos...")
for ano in anos_copa:
    separar_jogos_por_ciclo(ano)
print("Separação concluída!")

Iniciando a separação dos ciclos...
Ciclo 1930: 364 jogos salvos (De 1926-08-01 até 1930-06-30)
Ciclo 1934: 379 jogos salvos (De 1930-08-01 até 1934-05-31)
Ciclo 1938: 421 jogos salvos (De 1934-08-01 até 1938-05-31)
Ciclo 1950: 519 jogos salvos (De 1946-08-01 até 1950-05-31)
Ciclo 1954: 526 jogos salvos (De 1950-08-01 até 1954-05-31)
Ciclo 1958: 703 jogos salvos (De 1954-08-01 até 1958-05-31)
Ciclo 1962: 773 jogos salvos (De 1958-08-01 até 1962-05-31)
Ciclo 1966: 1081 jogos salvos (De 1962-08-01 até 1966-05-31)
Ciclo 1970: 1327 jogos salvos (De 1966-08-01 até 1970-05-31)
Ciclo 1974: 1634 jogos salvos (De 1970-08-01 até 1974-05-31)
Ciclo 1978: 1559 jogos salvos (De 1974-08-01 até 1978-05-31)
Ciclo 1982: 1796 jogos salvos (De 1978-08-01 até 1982-05-31)
Ciclo 1986: 1957 jogos salvos (De 1982-08-01 até 1986-05-31)
Ciclo 1990: 1917 jogos salvos (De 1986-08-01 até 1990-05-31)
Ciclo 1994: 2340 jogos salvos (De 1990-08-01 até 1994-05-31)
Ciclo 1998: 2960 jogos salvos (De 1994-08-01 até 1998-05

In [54]:
def stats_cicle(ano):
    
    df_confrontos = pd.read_csv(f"dados/confrontos/historico_confrontos_ciclo{ano}.csv")

    teams = list(set(df_confrontos["home_team"]).union(df_confrontos["away_team"]))

    print(len(teams))
    print(teams)
    

    df_stats = pd.DataFrame({"team": teams})
    df_stats = df_stats.set_index("team")
    df_stats["wins"] = 0
    df_stats["draws"] = 0
    df_stats["loses"] = 0
    df_stats["points"] = 0
    df_stats["games_played"] = 0

    for row in df_confrontos.itertuples():
        home, away = row.home_team, row.away_team
        home_score, away_score = row.home_score, row.away_score

        df_stats.at[home, "games_played"] += 1
        df_stats.at[away, "games_played"] += 1
        
        if home_score > away_score:
            df_stats.at[home, "wins"] += 1
            df_stats.at[away, "loses"] += 1
            
        elif away_score > home_score:
            df_stats.at[home, "loses"] += 1
            df_stats.at[away, "wins"] += 1
            
        else:
            df_stats.at[home, "draws"] += 1
            df_stats.at[away, "draws"] += 1

    df_stats["points"] = df_stats["wins"]*3 + df_stats["draws"]
    df_stats["points_per_game"] = round(df_stats["points"] / df_stats["games_played"],2)
    df_stats = df_stats.sort_values(by="points", ascending=False)
    df_stats.to_csv(f"dados/estatisticas_ciclos/estatisticas_ciclo{ano}.csv")

for ano in anos_copa:
    stats_cicle(ano)


66
['Finland', 'Lithuania', 'France', 'Trinidad and Tobago', 'Scotland', 'Hungary', 'Belgium', 'Luxembourg', 'Estonia', 'Argentina', 'Bolivia', 'Curaçao', 'Honduras', 'Spain', 'Sweden', 'Romania', 'Greece', 'Mexico', 'Barbados', 'Belarus', 'Norway', 'Catalonia', 'Italy', 'Cuba', 'Guatemala', 'China', 'Ukraine', 'England', 'Wales', 'Kenya', 'Costa Rica', 'Austria', 'Uruguay', 'Haiti', 'Czechoslovakia', 'Jersey', 'Netherlands', 'El Salvador', 'Nicaragua', 'Turkey', 'Egypt', 'Guernsey', 'Alderney', 'Philippines', 'Jamaica', 'Denmark', 'Aruba', 'Poland', 'Latvia', 'Germany', 'Republic of Ireland', 'Canada', 'Peru', 'United States', 'Paraguay', 'Chile', 'Japan', 'Basque Country', 'Northern Ireland', 'Guyana', 'Uganda', 'Yugoslavia', 'New Zealand', 'Bulgaria', 'Portugal', 'Switzerland']
63
['Finland', 'Lithuania', 'France', 'Indonesia', 'Trinidad and Tobago', 'Scotland', 'Hungary', 'Belgium', 'Luxembourg', 'Estonia', 'Argentina', 'Australia', 'Curaçao', 'Spain', 'Sweden', 'Romania', 'Greece'

In [55]:
def cicle_games(df, ano_copa):

    if ano_copa != 2022 and ano_copa != 2026:
        inicio_ciclo = date(year=ano_copa-4, month=8, day=1)
        fim_ciclo = date(year=ano_copa, month=5, day=26)
    
    elif ano_copa == 2022:
        inicio_ciclo = date(year=ano_copa-4, month=8, day=1)
        fim_ciclo = date(year=ano_copa, month=11, day=19)

    elif ano_copa == 2026:
        inicio_ciclo = date(year=ano_copa-4, month=12, day=20)
        fim_ciclo = date(year=ano_copa, month=6, day=10)

    selecoes = pd.read_csv(f'dados/copas/FIFA - {ano_copa}.csv')
    selecoes = selecoes['Team']
    jogos = df
    df_ciclo = jogos[(jogos['date'] >= inicio_ciclo) & (jogos['date'] <= fim_ciclo)]
    df_ciclo = df_ciclo[(df_ciclo['home_team'].isin(selecoes)) | (df_ciclo['away_team'].isin(selecoes))]
    df_ciclo = df_ciclo.reset_index()
    df_ciclo.to_csv(f"dados/confrontos/historico_confrontos_ciclo{ano_copa}.csv")
    stats_cicle(df, ano_copa)

for ano in anos_copa:
    try:
        cicle_games(df, ano)
    except: ...


In [56]:
#Separando os confrontos de cada copa

df_confrontos_copas = pd.read_csv("dados/confrontos_copas/WorldCupMatches.csv")
for ano in anos_copa:
    df_confrontos_copa = df_confrontos_copas[df_confrontos_copas["Year"] == ano]
    df_confrontos_copa["resultado"] = np.where(
        df_confrontos_copa["Home Team Goals"] > df_confrontos_copa["Away Team Goals"], 1, 
        np.where(df_confrontos_copa["Away Team Goals"] > df_confrontos_copa["Home Team Goals"], 2, 0)  
    )
    df_confrontos_copa.to_csv(f"dados/confrontos_copas/confrontos{ano}.csv")


/tmp/ipykernel_48507/3280243513.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_confrontos_copa["resultado"] = np.where(
/tmp/ipykernel_48507/3280243513.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_confrontos_copa["resultado"] = np.where(
/tmp/ipykernel_48507/3280243513.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

In [57]:
for ano in anos_copa:
    df_confrontos = pd.read_csv(f"dados/confrontos_copas/confrontos{ano}.csv")
    df_ciclo = pd.read_csv(f"dados/estatisticas_ciclos/estatisticas_ciclo{ano}.csv")
    df_input_modelo = pd.DataFrame()

    wins_difference = []
    draws_difference = []
    loses_difference = []
    points_per_game_differnce = []

    # Padroniza todas as colunas: minúsculas e com '_' no lugar do espaço
    df_confrontos.columns = df_confrontos.columns.str.lower().str.replace(' ', '_')


    for row in df_confrontos.itertuples():
        away_team, home_team  = row.away_team_goals, row.home_team_goals
        wins_difference.append(df_ciclo.loc[home_team]["wins"] - df_ciclo.loc[away_team]["wins"])
        draws_difference.append(df_ciclo.loc[home_team]["draws"] - df_ciclo.loc[away_team]["draws"])
        loses_difference.append(df_ciclo.loc[home_team]["loses"] - df_ciclo.loc[away_team]["loses"])
        points_per_game_differnce.append(df_ciclo.loc[home_team]["points_per_game"] - df_ciclo.loc[away_team]["points_per_game"])
    
    df_input_modelo["wins_difference"] = wins_difference
    df_input_modelo["draws_difference"] = draws_difference
    df_input_modelo["loses_difference"] = loses_difference
    df_input_modelo["points_per_game_difference"] = points_per_game_differnce
    df_input_modelo["result"] = df_confrontos["resultado"]

    df_input_modelo.to_csv(f"dados/dados_modelo/input_modelo{ano}.csv")